### Investigating concordance of low divergence clades with cenhap labels 

In [3]:
import os
import pandas as pd

base_dir = '/private/groups/migalab/juklucas/centrolign/cenhap_assignment/cenhap_inference_out'

dfs = []
for chrom in os.listdir(base_dir):
    fpath = os.path.join(base_dir, chrom, f'{chrom}.cenhap_predictions.tsv')
    if os.path.isfile(fpath):
        tmp = pd.read_csv(fpath, sep='\t')
        tmp['chrom'] = chrom
        dfs.append(tmp)

cenhap_df = pd.concat(dfs, ignore_index=True)
cenhap_df.columns = ['sample', 'cenhap_assignment', 'chrom']

clade_dir = '/private/groups/patenlab/mira/centrolign/analysis/low_divergence_clades/max_dist_0.8_min_pairwise_0.95'

clade_dfs = []
for chrom in os.listdir(clade_dir):
    fpath = os.path.join(clade_dir, chrom, '_clades.csv')
    if os.path.isfile(fpath):
        tmp = pd.read_csv(fpath)
        tmp['chrom'] = chrom
        clade_dfs.append(tmp)

clade_df = pd.concat(clade_dfs, ignore_index=True)
clade_df.columns = ['clade_assignment', 'sample', 'chrom']

cenhap_df = cenhap_df.merge(clade_df, on=['sample', 'chrom'], how='left')
cenhap_df['clade_assignment'] = cenhap_df['clade_assignment'].fillna('NA')

cenhap_df = cenhap_df[cenhap_df['clade_assignment'] != 'NA']

cenhap_df.head()



,sample,cenhap_assignment,chrom,clade_assignment
1,HG00097.1,3,chr11,Clade_18
2,HG00097.2,3,chr11,Clade_15
3,HG00099.1,2,chr11,Clade_5
5,HG00126.1,2,chr11,Clade_5
6,HG00126.2,3,chr11,Clade_18


In [4]:
for chrom, grp in cenhap_df.groupby('chrom'):
    n_cenhaps = grp['cenhap_assignment'].nunique()
    n_clades  = grp['clade_assignment'].nunique()
    print(f"{chrom}: {n_cenhaps} cenhap groups, {n_clades} clade groups")


chr10: 18 cenhap groups, 24 clade groups
chr11: 16 cenhap groups, 18 clade groups
chr12: 8 cenhap groups, 21 clade groups
chr17: 11 cenhap groups, 20 clade groups
chr4: 7 cenhap groups, 10 clade groups
chr6: 6 cenhap groups, 17 clade groups
chr9: 14 cenhap groups, 18 clade groups


In [5]:
# examine concordance 
tricky = {
    'chr4':  ['HG00738.1'],
    'chr6':  ['HG02293.1', 'HG01123.1', 'HG02148.2', 'HG01975.2'],
    'chr9':  ['HG02293.1', 'NA21144.2', 'HG00099.1', 'HG01981.1'],
    'chr10': ['HG02698.1', 'NA18948.1', 'HG02132.2', 'HG03521.1'],
    'chr11': ['HG01099.2', 'NA18608.2'],
    'chr12': ['HG01978.2', 'HG00280.1', 'HG02083.1', 'HG03017.1'],
    'chr17': ['HG03225.1', 'HG02668.2'],
}

for chrom, samples in tricky.items():
    chrom_df = cenhap_df[cenhap_df['chrom'] == chrom]
    print(f"\n── {chrom} ──")
    for sample in samples:
        row = chrom_df[chrom_df['sample'] == sample]
        if row.empty:
            print(f"  {sample}: not found")
            continue

        cenhap = row['cenhap_assignment'].values[0]
        clade  = row['clade_assignment'].values[0]

        cenhap_members = set(chrom_df[chrom_df['cenhap_assignment'] == cenhap]['sample'])
        clade_members  = set(chrom_df[chrom_df['clade_assignment']  == clade ]['sample'])
        overlap        = cenhap_members & clade_members

        pct_of_cenhap = 100 * len(overlap) / len(cenhap_members)
        pct_of_clade  = 100 * len(overlap) / len(clade_members)

        print(f"  {sample}")
        print(f"    cenhap: {cenhap} ({len(cenhap_members)} samples)")
        print(f"    clade:  {clade}  ({len(clade_members)} samples)")
        print(f"    overlap: {len(overlap)} samples")
        print(f"    overlap as % of cenhap: {pct_of_cenhap:.1f}%")
        print(f"    overlap as % of clade:  {pct_of_clade:.1f}%")



── chr4 ──
  HG00738.1
    cenhap: 6 (2 samples)
    clade:  Clade_7  (2 samples)
    overlap: 2 samples
    overlap as % of cenhap: 100.0%
    overlap as % of clade:  100.0%

── chr6 ──
  HG02293.1
    cenhap: 1 (107 samples)
    clade:  Clade_16  (4 samples)
    overlap: 4 samples
    overlap as % of cenhap: 3.7%
    overlap as % of clade:  100.0%
  HG01123.1
    cenhap: 1 (107 samples)
    clade:  Clade_16  (4 samples)
    overlap: 4 samples
    overlap as % of cenhap: 3.7%
    overlap as % of clade:  100.0%
  HG02148.2
    cenhap: 1 (107 samples)
    clade:  Clade_16  (4 samples)
    overlap: 4 samples
    overlap as % of cenhap: 3.7%
    overlap as % of clade:  100.0%
  HG01975.2
    cenhap: 1 (107 samples)
    clade:  Clade_16  (4 samples)
    overlap: 4 samples
    overlap as % of cenhap: 3.7%
    overlap as % of clade:  100.0%

── chr9 ──
  HG02293.1
    cenhap: 3 (121 samples)
    clade:  Clade_18  (120 samples)
    overlap: 120 samples
    overlap as % of cenhap: 99.2%
    o

```sh
chromosomes=("chr4" "chr6" "chr9" "chr10" "chr11" "chr12" "chr17")
for chr in "${chromosomes[@]}"
do
    python3 /Users/miramastoras/Desktop/github_repos/centrolign_analysis/scripts/pairwise_tree_heatmap_v2.py \
        -t /Users/miramastoras/Desktop/HPRC_release2_QCv2_all_pairs_heatmaps/${chr}_r2_QC_v2_centrolign_all_pairs_nj_tree.format5.nwk \
        -s /Users/miramastoras/Desktop/HPRC_release2_QCv2_all_pairs_heatmaps/${chr}.samples.txt \
        -p /Users/miramastoras/Desktop/HPRC_release2_QCv2_all_pairs_heatmaps/${chr}_r2_QC_v2_centrolign_pairwise_distance.csv \
        -m "Centrolign all pairs distances" \
        -n "${chr} NJ tree" \
        -d "Pairwise match distance" \
        -o /Users/miramastoras/Desktop/cenhap_tricky_${chr} --no_labels \
        --highlight_samples /Users/miramastoras/Desktop/${chr}_tricky_highlight.csv
done

docker run -u `id -u`:`id -g` -v /private/groups:/private/groups/ \
        miramastoras/tree_heatmap:latest \
        python3 /private/groups/patenlab/mira/centrolign/github/centrolign_analysis/scripts/pairwise_tree_heatmap_v2.py \
        /private/groups/patenlab/mira/centrolign/batch_submissions/centrolign/release2_QC_v2/all_pairs/${chr}/pairwise_cigar/ \
        > /private/groups/patenlab/mira/centrolign/batch_submissions/centrolign/release2_QC_v2/all_pairs/nj_trees/${chr}_r2_QC_v2_centrolign_all_pairs_nj_tree.nwk
```